# DEE — Test materia con N grande (v2: jitter calibrado a SIM 19)

**Cambios respecto a v1:**
1. **jitter = 0.01** (igual a SIM 19) en lugar de 0.05
2. **Clip mínimo de distancias en 0.3·a** para evitar singularidades del kernel 1/d²
3. **Defectos definidos por percentiles** (top/bottom 5%) en lugar de ±2σ — más robusto a colas asimétricas
4. **Validación contra σ_F de SIM 19 (~0.107)** al inicio: si el primer N=2000 da σ muy distinto, alertar
5. **Bootstrap explícito** sobre réplicas para errores estadísticos limpios

**Tiempo estimado**: 3-4 horas en Colab Pro CPU

**RAM**: ~13 GB (compatible con Colab Pro estándar, igual que v1)

## 0. Setup

In [ ]:
import os, sys, time, json, pickle, gc, warnings
import numpy as np
from scipy.linalg import eigh
from scipy.sparse.linalg import eigsh
from scipy.sparse import csr_matrix, diags
from scipy.stats import linregress, chi2 as chi2_dist
from scipy.optimize import curve_fit
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

OUTPUT_DIR = '/content/dee_materia_v2'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(f'{OUTPUT_DIR}/figuras', exist_ok=True)
os.makedirs(f'{OUTPUT_DIR}/datos', exist_ok=True)
os.makedirs(f'{OUTPUT_DIR}/logs', exist_ok=True)

LOG_FILE = f'{OUTPUT_DIR}/logs/ejecucion.log'
log_handle = open(LOG_FILE, 'w')

def log(msg):
    print(msg)
    log_handle.write(msg + '\n')
    log_handle.flush()

from IPython.display import display, Javascript
try:
    display(Javascript('''
    function ClickConnect(){
        document.querySelector("colab-connect-button").shadowRoot
                .querySelector("#connect").click();
    }
    setInterval(ClickConnect, 60000);
    '''))
    log("Keep-alive activo")
except Exception:
    log("Keep-alive no disponible")

log("=" * 70)
log("DEE — TEST MATERIA v2 (jitter=0.01 calibrado a SIM 19)")
log("=" * 70)
log(f"Inicio: {time.strftime('%Y-%m-%d %H:%M:%S')}")

try:
    import psutil
    mem_gb = psutil.virtual_memory().total / 1e9
    log(f"RAM total: {mem_gb:.1f} GB")
except ImportError:
    pass

# Constantes globales clave
JITTER = 0.01            # Calibrado a SIM 19 (era 0.05 en v1)
CLIP_FACTOR = 0.3        # Distancia mínima permitida = CLIP_FACTOR · a
DEFECT_PERCENTILE = 5    # Top/bottom 5% son defectos (más robusto que ±2σ)
TARGET_SIGMA_F_SIM19 = 0.107  # Si σ_F del cálculo difiere mucho, alerta

log(f"\nParámetros calibrados:")
log(f"  jitter = {JITTER} (igual a SIM 19)")
log(f"  clip mínimo distancia = {CLIP_FACTOR} · a")
log(f"  umbral defecto = {DEFECT_PERCENTILE}% por percentil")
log(f"  σ_F esperado (de SIM 19): {TARGET_SIGMA_F_SIM19}")

## 1. Funciones core CON CORRECCIONES

In [ ]:
def init_cristal_jiterado(N, L=1.0, jitter=JITTER, seed=42):
    rng = np.random.default_rng(seed)
    n_side = int(np.ceil(N**(1/3)))
    pts = []
    for i in range(n_side):
        for j in range(n_side):
            for k in range(n_side):
                pts.append([i/n_side, j/n_side, k/n_side])
    pts = np.array(pts[:N]) * L
    pts += rng.normal(0, jitter, pts.shape)
    return pts % L

def compute_distances_pbc_clipped(points, L=1.0, clip_min=None):
    """Distancias con clip mínimo para evitar singularidades del kernel.
    
    El clip introduce sesgo despreciable si clip_min << a (spacing típico),
    pero elimina los pares patológicos que dominaban σ(F) en v1.
    """
    diff = points[:, None, :] - points[None, :, :]
    diff = diff - L * np.round(diff / L)
    dist = np.linalg.norm(diff, axis=-1)
    if clip_min is not None:
        np.fill_diagonal(dist, np.inf)
        dist = np.maximum(dist, clip_min)
        np.fill_diagonal(dist, np.inf)  # Re-aplicar diagonal infinita
    return dist

def build_laplacian_dense(points, k_neighbors=30, L=1.0):
    N = len(points)
    a = N**(-1/3)
    clip = CLIP_FACTOR * a
    
    dist = compute_distances_pbc_clipped(points, L, clip_min=clip)
    
    knn_idx = np.argpartition(dist, k_neighbors, axis=1)[:, :k_neighbors]
    W = np.zeros((N, N))
    for i in range(N):
        for j in knn_idx[i]:
            d = dist[i, j]
            w = 1.0 / d**2
            W[i, j] = w
            W[j, i] = w
    return np.diag(W.sum(axis=1)) - W

def build_laplacian_sparse(points, k_neighbors=30, L=1.0):
    """Sparse para N grande, con mismo clip."""
    N = len(points)
    a = N**(-1/3)
    clip = CLIP_FACTOR * a
    
    block_size = 1000
    rows, cols, weights = [], [], []
    
    for start in range(0, N, block_size):
        end = min(start + block_size, N)
        diff = points[start:end, None, :] - points[None, :, :]
        diff = diff - L * np.round(diff / L)
        dist_block = np.linalg.norm(diff, axis=-1)
        
        for i_local in range(end - start):
            i_global = start + i_local
            dist_block[i_local, i_global] = np.inf
        
        # Aplicar clip
        dist_block = np.maximum(dist_block, clip)
        # Re-poner diagonal infinita
        for i_local in range(end - start):
            i_global = start + i_local
            dist_block[i_local, i_global] = np.inf
        
        knn = np.argpartition(dist_block, k_neighbors, axis=1)[:, :k_neighbors]
        for i_local in range(end - start):
            i_global = start + i_local
            for j in knn[i_local]:
                d = dist_block[i_local, j]
                w = 1.0 / d**2
                rows.append(i_global)
                cols.append(j)
                weights.append(w)
    
    rows = np.array(rows); cols = np.array(cols); weights = np.array(weights)
    W_sparse = csr_matrix((weights, (rows, cols)), shape=(N, N))
    W_sym = W_sparse.maximum(W_sparse.T)
    D = diags(np.array(W_sym.sum(axis=1)).flatten())
    return D - W_sym

def compute_F_local(points, k_neighbors=30, L=1.0):
    """F_local con clip mínimo aplicado."""
    N = len(points)
    a = N**(-1/3)
    clip = CLIP_FACTOR * a
    
    block_size = 2000
    F = np.zeros(N)
    for start in range(0, N, block_size):
        end = min(start + block_size, N)
        diff = points[start:end, None, :] - points[None, :, :]
        diff = diff - L * np.round(diff / L)
        dist_block = np.linalg.norm(diff, axis=-1)
        
        for i_local in range(end - start):
            i_global = start + i_local
            dist_block[i_local, i_global] = np.inf
        
        dist_block = np.maximum(dist_block, clip)
        for i_local in range(end - start):
            i_global = start + i_local
            dist_block[i_local, i_global] = np.inf
        
        knn = np.argpartition(dist_block, k_neighbors, axis=1)[:, :k_neighbors]
        for i_local in range(end - start):
            i_global = start + i_local
            ds = dist_block[i_local, knn[i_local]]
            F[i_global] = (1.0 / ds**2).sum()
    return F / F.mean()

log("Funciones core con correcciones listas.")

## 2. Funciones de análisis con definición robusta de defectos

In [ ]:
def diagonalize_laplacian(Lap, N, mode='dense', n_low=200, n_high=200):
    if mode == 'dense':
        if hasattr(Lap, 'toarray'):
            Lap = Lap.toarray()
        eigvals, eigvecs = eigh(Lap)
        iprs = np.sum(eigvecs**4, axis=0)
        centers = np.argmax(eigvecs**2, axis=0)
        return eigvals, iprs, centers, 'all'
    elif mode == 'lanczos':
        eigvals_low, eigvecs_low = eigsh(Lap, k=n_low, which='SM', maxiter=5000)
        eigvals_high, eigvecs_high = eigsh(Lap, k=n_high, which='LM', maxiter=5000)
        all_eigvals = np.concatenate([eigvals_low, eigvals_high])
        all_eigvecs = np.concatenate([eigvecs_low, eigvecs_high], axis=1)
        sort_idx = np.argsort(all_eigvals)
        eigvals_sorted = all_eigvals[sort_idx]
        eigvecs_sorted = all_eigvecs[:, sort_idx]
        iprs = np.sum(eigvecs_sorted**4, axis=0)
        centers = np.argmax(eigvecs_sorted**2, axis=0)
        return eigvals_sorted, iprs, centers, 'partial'

def caracterize_defects_percentile(F_local, points, percentile=DEFECT_PERCENTILE, L=1.0):
    """Defectos por percentil — robusto a colas asimétricas.
    
    Defectos low: F < percentil_inferior
    Defectos high: F > percentil_superior
    
    Asegura siempre el mismo número de defectos (~5% × 2 = 10% del total).
    """
    threshold_low = np.percentile(F_local, percentile)
    threshold_high = np.percentile(F_local, 100 - percentile)
    
    is_low = F_local < threshold_low
    is_high = F_local > threshold_high
    
    N = len(points)
    a = N**(-1/3)
    
    shell_edges = np.array([0.5*a, 1.5*a, 2.5*a, 3.5*a, 4.5*a, 6*a, 8*a])
    shell_centers = 0.5 * (shell_edges[:-1] + shell_edges[1:])
    
    delta_F = F_local - F_local.mean()
    
    def_low_idx = np.where(is_low)[0]
    def_high_idx = np.where(is_high)[0]
    
    # Submuestreo si hay demasiados
    max_for_profile = 80
    rng = np.random.default_rng(0)
    if len(def_low_idx) > max_for_profile:
        def_low_idx = rng.choice(def_low_idx, max_for_profile, replace=False)
    if len(def_high_idx) > max_for_profile:
        def_high_idx = rng.choice(def_high_idx, max_for_profile, replace=False)
    
    profiles_low = []
    for idx in def_low_idx:
        diff = points - points[idx]
        diff = diff - L * np.round(diff / L)
        dists = np.linalg.norm(diff, axis=-1)
        profile = []
        for i in range(len(shell_edges) - 1):
            mask = (dists >= shell_edges[i]) & (dists < shell_edges[i+1])
            mask[idx] = False
            profile.append(delta_F[mask].mean() if mask.sum() > 0 else np.nan)
        profiles_low.append(profile)
    
    profiles_high = []
    for idx in def_high_idx:
        diff = points - points[idx]
        diff = diff - L * np.round(diff / L)
        dists = np.linalg.norm(diff, axis=-1)
        profile = []
        for i in range(len(shell_edges) - 1):
            mask = (dists >= shell_edges[i]) & (dists < shell_edges[i+1])
            mask[idx] = False
            profile.append(delta_F[mask].mean() if mask.sum() > 0 else np.nan)
        profiles_high.append(profile)
    
    profiles_low = np.array(profiles_low) if profiles_low else np.array([]).reshape(0, len(shell_centers))
    profiles_high = np.array(profiles_high) if profiles_high else np.array([]).reshape(0, len(shell_centers))
    
    # Ajuste exponencial usando MEDIANA (robusto a outliers)
    def fit_exp_robust(centers, profile):
        if profile.shape[0] < 3:
            return None
        # MEDIANA en lugar de media para robustez
        median_p = np.nanmedian(profile, axis=0)
        abs_p = np.abs(median_p)
        valid = ~np.isnan(abs_p) & (abs_p > 1e-5)
        if valid.sum() < 3:
            return None
        try:
            # Bound xi: debe ser positivo y < L/2
            popt, _ = curve_fit(
                lambda r, A, xi: A * np.exp(-r/xi),
                centers[valid], abs_p[valid],
                p0=[abs_p[valid][0], 0.05],
                bounds=([0, 0.001], [10, 0.5]),
                maxfev=5000
            )
            return popt[1]
        except Exception:
            return None
    
    xi_low = fit_exp_robust(shell_centers, profiles_low)
    xi_high = fit_exp_robust(shell_centers, profiles_high)
    
    return {
        'mean_F': float(F_local.mean()),
        'sigma_F': float(F_local.std()),
        'threshold_low': float(threshold_low),
        'threshold_high': float(threshold_high),
        'frac_low': float(is_low.sum() / N),
        'frac_high': float(is_high.sum() / N),
        'shell_centers_in_a': (shell_centers / a).tolist(),
        'profile_low_median': np.nanmedian(profiles_low, axis=0).tolist() if profiles_low.shape[0] > 0 else [],
        'profile_high_median': np.nanmedian(profiles_high, axis=0).tolist() if profiles_high.shape[0] > 0 else [],
        'profile_low_mean': np.nanmean(profiles_low, axis=0).tolist() if profiles_low.shape[0] > 0 else [],
        'profile_high_mean': np.nanmean(profiles_high, axis=0).tolist() if profiles_high.shape[0] > 0 else [],
        'xi_def_low_in_a': float(xi_low / a) if xi_low else None,
        'xi_def_high_in_a': float(xi_high / a) if xi_high else None,
        'is_low': is_low,
        'is_high': is_high,
    }

def find_mobility_edge(eigvals, iprs, n_bands=20):
    nonzero = eigvals > 1e-6
    omegas = np.sqrt(eigvals[nonzero])
    iprs_nz = iprs[nonzero]
    
    omega_bands = np.linspace(omegas.min(), omegas.max(), n_bands + 1)
    band_centers = []
    band_ipr_med = []
    for i in range(n_bands):
        mask = (omegas >= omega_bands[i]) & (omegas < omega_bands[i+1])
        if mask.sum() >= 5:
            band_centers.append(0.5 * (omega_bands[i] + omega_bands[i+1]))
            band_ipr_med.append(np.median(iprs_nz[mask]))
    band_centers = np.array(band_centers)
    band_ipr_med = np.array(band_ipr_med)
    
    if len(band_ipr_med) == 0:
        return None, band_centers, band_ipr_med
    ipr_min_obs = band_ipr_med.min()
    threshold = 5 * ipr_min_obs
    
    for i, ipr in enumerate(band_ipr_med):
        if ipr > threshold:
            return float(band_centers[i]), band_centers, band_ipr_med
    return None, band_centers, band_ipr_med

def correlation_modes_defects(eigvals, iprs, centers, F_local, threshold_low, threshold_high,
                                ipr_threshold=0.10):
    is_low = F_local < threshold_low
    is_high = F_local > threshold_high
    
    localized_mask = iprs > ipr_threshold
    n_loc = int(localized_mask.sum())
    if n_loc == 0:
        return None
    
    centers_loc = centers[localized_mask]
    n_low = int(is_low[centers_loc].sum())
    n_high = int(is_high[centers_loc].sum())
    n_norm = n_loc - n_low - n_high
    
    N = len(F_local)
    frac_low_pop = is_low.sum() / N
    frac_high_pop = is_high.sum() / N
    frac_norm_pop = 1 - frac_low_pop - frac_high_pop
    
    enr_low = (n_low / n_loc) / frac_low_pop if frac_low_pop > 0 else None
    enr_high = (n_high / n_loc) / frac_high_pop if frac_high_pop > 0 else None
    
    expected_low = n_loc * frac_low_pop
    expected_high = n_loc * frac_high_pop
    expected_norm = n_loc * frac_norm_pop
    chi2 = 0
    for obs, exp in [(n_low, expected_low), (n_high, expected_high), (n_norm, expected_norm)]:
        if exp > 0:
            chi2 += (obs - exp)**2 / exp
    p_value = 1 - chi2_dist.cdf(chi2, df=2)
    
    omegas_loc = np.sqrt(np.maximum(eigvals[localized_mask], 0))
    omega_on_low = omegas_loc[is_low[centers_loc]]
    omega_on_high = omegas_loc[is_high[centers_loc]]
    omega_on_norm = omegas_loc[~is_low[centers_loc] & ~is_high[centers_loc]]
    
    return {
        'n_localized_modes': n_loc,
        'centered_on_low': n_low,
        'centered_on_high': n_high,
        'centered_on_normal': n_norm,
        'enrichment_low': float(enr_low) if enr_low is not None else None,
        'enrichment_high': float(enr_high) if enr_high is not None else None,
        'chi2': float(chi2),
        'p_value': float(p_value),
        'mean_omega_on_low': float(omega_on_low.mean()) if len(omega_on_low) > 0 else None,
        'mean_omega_on_high': float(omega_on_high.mean()) if len(omega_on_high) > 0 else None,
        'mean_omega_on_normal': float(omega_on_norm.mean()) if len(omega_on_norm) > 0 else None,
    }

log("Funciones de análisis listas.")

## 3. VALIDACIÓN INICIAL: ¿σ_F coincide con SIM 19?

Antes de empezar el barrido completo, una réplica piloto N=2000 para verificar que la corrección de jitter produce σ_F ≈ 0.107 (valor de SIM 19). Si difiere mucho, hay que reconsiderar el setup.

In [ ]:
log("\n" + "="*70)
log("VALIDACIÓN INICIAL: ¿σ_F coincide con SIM 19?")
log("="*70)
log(f"\nValor objetivo (SIM 19, N=500, annealing): σ_F = {TARGET_SIGMA_F_SIM19:.4f}")
log("Verificamos si jitter=0.01 + clip lo reproduce con N=2000 cristal jiterado.")

t_pilot = time.time()
points_pilot = init_cristal_jiterado(2000, jitter=JITTER, seed=12345)
F_pilot = compute_F_local(points_pilot)
sigma_pilot = F_pilot.std()
log(f"\nN=2000 piloto: ⟨F⟩ = {F_pilot.mean():.4f}, σ = {sigma_pilot:.4f}")
log(f"Tiempo: {time.time()-t_pilot:.1f}s")

ratio = sigma_pilot / TARGET_SIGMA_F_SIM19
log(f"\nRatio σ_pilot / σ_SIM19 = {ratio:.2f}")
if 0.5 < ratio < 2.0:
    log("✓ σ_F está en rango razonable (factor 0.5-2× de SIM 19)")
    log("  El jitter calibrado funciona bien. Procedemos.")
elif ratio > 2.0:
    log(f"⚠ σ_F demasiado alto. La fenomenología puede diferir de SIM 19.")
    log("  Procedemos pero la comparación con datos del documento será cualitativa.")
else:
    log("⚠ σ_F demasiado bajo. Cristal demasiado uniforme.")

# También: histograma rápido de F_pilot para entender la distribución
log(f"\nDistribución de F_pilot:")
log(f"  Min:    {F_pilot.min():.4f}")
log(f"  Max:    {F_pilot.max():.4f}")
log(f"  P5:     {np.percentile(F_pilot, 5):.4f}")
log(f"  P95:    {np.percentile(F_pilot, 95):.4f}")
log(f"  P99:    {np.percentile(F_pilot, 99):.4f}")
log(f"  Skewness (asimetría): {(F_pilot - F_pilot.mean()).mean()/sigma_pilot**3 if sigma_pilot > 0 else 0:.3f}")

del points_pilot, F_pilot
gc.collect()

## 4. Barrido principal en N (igual estructura que v1)

In [ ]:
config = [
    {'N': 2000, 'replicas': 6, 'mode': 'dense', 'k': 30},
    {'N': 4000, 'replicas': 4, 'mode': 'dense', 'k': 30},
    {'N': 8000, 'replicas': 2, 'mode': 'lanczos', 'k': 30, 'n_low': 200, 'n_high': 200},
    {'N': 16000, 'replicas': 2, 'mode': 'lanczos', 'k': 30, 'n_low': 300, 'n_high': 300},
]

log("\n" + "="*70)
log("BARRIDO PRINCIPAL")
log("="*70)
for c in config:
    log(f"  N={c['N']}, {c['replicas']} réplicas, modo={c['mode']}")

all_results = {}

for cfg in config:
    N = cfg['N']
    n_rep = cfg['replicas']
    mode = cfg['mode']
    k_nn = cfg['k']
    
    log(f"\n{'='*60}")
    log(f"N = {N}, modo = {mode}")
    log(f"{'='*60}")
    
    results_N = []
    
    for r in range(n_rep):
        seed = 100 * (r + 1) + 1
        log(f"\n--- Réplica {r+1}/{n_rep}, seed={seed} ---")
        t_rep = time.time()
        
        points = init_cristal_jiterado(N, jitter=JITTER, seed=seed)
        
        t1 = time.time()
        F_local = compute_F_local(points, k_neighbors=k_nn)
        log(f"  F_local: σ={F_local.std():.4f} (target ~0.1), {time.time()-t1:.1f}s")
        
        t1 = time.time()
        def_data = caracterize_defects_percentile(F_local, points, percentile=DEFECT_PERCENTILE)
        log(f"  Test 1: defectos low={def_data['frac_low']*100:.1f}% (P{DEFECT_PERCENTILE}), "
            f"high={def_data['frac_high']*100:.1f}% (P{100-DEFECT_PERCENTILE})")
        log(f"          ξ_low={def_data['xi_def_low_in_a']}, ξ_high={def_data['xi_def_high_in_a']}, "
            f"t={time.time()-t1:.1f}s")
        
        t1 = time.time()
        if mode == 'dense':
            Lap = build_laplacian_dense(points, k_neighbors=k_nn)
        else:
            Lap = build_laplacian_sparse(points, k_neighbors=k_nn)
        log(f"  Laplaciano: {time.time()-t1:.1f}s")
        
        t1 = time.time()
        if mode == 'dense':
            eigvals, iprs, centers, _ = diagonalize_laplacian(Lap, N, mode='dense')
        else:
            eigvals, iprs, centers, _ = diagonalize_laplacian(
                Lap, N, mode='lanczos',
                n_low=cfg.get('n_low', 200), n_high=cfg.get('n_high', 200)
            )
        log(f"  Diag ({mode}): {len(eigvals)} eigvals, {time.time()-t1:.1f}s")
        
        mobility_edge, band_centers, band_ipr_med = find_mobility_edge(eigvals, iprs)
        log(f"  Test 5: mobility edge ω = {mobility_edge}")
        
        corr = correlation_modes_defects(
            eigvals, iprs, centers, F_local,
            def_data['threshold_low'], def_data['threshold_high']
        )
        if corr is not None:
            log(f"  Test 3: enr_low={corr['enrichment_low']}, enr_high={corr['enrichment_high']}, "
                f"p={corr['p_value']:.2e}")
        
        nonzero = eigvals > 1e-6
        omegas = np.sqrt(eigvals[nonzero])
        
        result = {
            'N': N, 'replica': r, 'seed': seed,
            'F_local_mean': float(F_local.mean()),
            'F_local_std': float(F_local.std()),
            'frac_low': def_data['frac_low'],
            'frac_high': def_data['frac_high'],
            'threshold_low': def_data['threshold_low'],
            'threshold_high': def_data['threshold_high'],
            'xi_def_low_in_a': def_data['xi_def_low_in_a'],
            'xi_def_high_in_a': def_data['xi_def_high_in_a'],
            'profile_low_median': def_data['profile_low_median'],
            'profile_high_median': def_data['profile_high_median'],
            'shell_centers_in_a': def_data['shell_centers_in_a'],
            'mobility_edge': mobility_edge,
            'band_centers': band_centers.tolist() if isinstance(band_centers, np.ndarray) else band_centers,
            'band_ipr_med': band_ipr_med.tolist() if isinstance(band_ipr_med, np.ndarray) else band_ipr_med,
            'omega_min_observed': float(omegas.min()),
            'omega_max_observed': float(omegas.max()),
            'ipr_min_observed': float(iprs[nonzero].min()),
            'ipr_max_observed': float(iprs[nonzero].max()),
            'mode': mode,
            'n_eigvals_computed': len(eigvals),
        }
        if corr is not None:
            result['correlation'] = corr
        
        if mode == 'dense' and len(omegas) > 1000:
            sample_idx = np.linspace(0, len(omegas)-1, 1000, dtype=int)
            result['omega_sample'] = omegas[sample_idx].tolist()
            result['ipr_sample'] = iprs[nonzero][sample_idx].tolist()
        else:
            result['omega_sample'] = omegas.tolist()
            result['ipr_sample'] = iprs[nonzero].tolist()
        
        results_N.append(result)
        log(f"  Réplica completa en {time.time()-t_rep:.1f}s")
        
        del Lap, eigvals, iprs, centers, points, F_local
        gc.collect()
    
    all_results[N] = results_N
    
    with open(f'{OUTPUT_DIR}/datos/checkpoint_N{N}.pkl', 'wb') as f:
        pickle.dump(all_results, f)
    log(f"  ✓ Checkpoint N={N} guardado")

log("\n=== Barrido principal completado ===")

## 5. Análisis con bootstrap sobre réplicas

In [ ]:
log("\n" + "="*70)
log("ANÁLISIS CON BOOTSTRAP")
log("="*70)

def bootstrap_stat(values, stat_fn=np.mean, n_boot=1000, seed=42):
    """Bootstrap simple sobre las réplicas."""
    rng = np.random.default_rng(seed)
    values = np.array([v for v in values if v is not None and not np.isnan(v)])
    if len(values) < 2:
        return None, None, None, None
    boots = []
    for _ in range(n_boot):
        sample = rng.choice(values, len(values), replace=True)
        boots.append(stat_fn(sample))
    boots = np.array(boots)
    return float(stat_fn(values)), float(boots.std()), float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5))

summary_per_N = {}
log(f"\n{'N':>8} {'σ(F) ± SE':>16} {'%low':>8} {'%high':>8} {'ξ_low ± SE':>16} {'ξ_high ± SE':>16} {'mob.edge ± SE':>17} {'enr_high ± SE':>17}")
log("-" * 130)

for N in sorted(all_results.keys()):
    results = all_results[N]
    if not results:
        continue
    
    sigma_Fs = [r['F_local_std'] for r in results]
    frac_lows = [r['frac_low'] for r in results]
    frac_highs = [r['frac_high'] for r in results]
    xi_lows = [r['xi_def_low_in_a'] for r in results if r['xi_def_low_in_a'] is not None]
    xi_highs = [r['xi_def_high_in_a'] for r in results if r['xi_def_high_in_a'] is not None]
    mob_edges = [r['mobility_edge'] for r in results if r['mobility_edge'] is not None]
    enr_lows = [r['correlation']['enrichment_low'] for r in results 
                if 'correlation' in r and r['correlation']['enrichment_low'] is not None]
    enr_highs = [r['correlation']['enrichment_high'] for r in results 
                 if 'correlation' in r and r['correlation']['enrichment_high'] is not None]
    
    sigma_F_m, sigma_F_se, _, _ = bootstrap_stat(sigma_Fs)
    xi_low_m, xi_low_se, xi_low_lo, xi_low_hi = bootstrap_stat(xi_lows) if xi_lows else (None, None, None, None)
    xi_high_m, xi_high_se, xi_high_lo, xi_high_hi = bootstrap_stat(xi_highs) if xi_highs else (None, None, None, None)
    mob_m, mob_se, _, _ = bootstrap_stat(mob_edges) if mob_edges else (None, None, None, None)
    enr_low_m, enr_low_se, _, _ = bootstrap_stat(enr_lows) if enr_lows else (None, None, None, None)
    enr_high_m, enr_high_se, _, _ = bootstrap_stat(enr_highs) if enr_highs else (None, None, None, None)
    
    summary_per_N[N] = {
        'F_std_mean': sigma_F_m, 'F_std_se': sigma_F_se,
        'frac_low_mean': float(np.mean(frac_lows)),
        'frac_high_mean': float(np.mean(frac_highs)),
        'xi_def_low_mean': xi_low_m, 'xi_def_low_se': xi_low_se,
        'xi_def_high_mean': xi_high_m, 'xi_def_high_se': xi_high_se,
        'mobility_edge_mean': mob_m, 'mobility_edge_se': mob_se,
        'enrichment_low_mean': enr_low_m, 'enrichment_low_se': enr_low_se,
        'enrichment_high_mean': enr_high_m, 'enrichment_high_se': enr_high_se,
        'n_replicas': len(results),
    }
    
    s = summary_per_N[N]
    def fmt(m, se, w=14):
        if m is None:
            return f"{'—':>{w}}"
        return f"{m:>{w-6}.3f} ± {se:.3f}" if se is not None else f"{m:>{w}.3f}"
    
    log(f"{N:>8} {fmt(s['F_std_mean'], s['F_std_se']):>16} "
        f"{s['frac_low_mean']*100:>6.1f}%  {s['frac_high_mean']*100:>6.1f}%  "
        f"{fmt(s['xi_def_low_mean'], s['xi_def_low_se']):>16} "
        f"{fmt(s['xi_def_high_mean'], s['xi_def_high_se']):>16} "
        f"{fmt(s['mobility_edge_mean'], s['mobility_edge_se']):>17} "
        f"{fmt(s['enrichment_high_mean'], s['enrichment_high_se']):>17}")

# Extrapolación a N → ∞ con error estadístico
log("\n=== Extrapolación N → ∞ ===")
Ns_arr = np.array(sorted(summary_per_N.keys()))
x_inv = 1.0 / Ns_arr**(1/3)

for key, label in [
    ('xi_def_low_mean', 'ξ_def (low)'),
    ('xi_def_high_mean', 'ξ_def (high)'),
    ('mobility_edge_mean', 'mobility edge ω'),
    ('enrichment_high_mean', 'enrichment high'),
    ('F_std_mean', 'σ(F_local)'),
]:
    vals = []
    xs = []
    for i, N in enumerate(Ns_arr):
        v = summary_per_N[N].get(key)
        if v is not None:
            vals.append(float(v))
            xs.append(x_inv[i])
    if len(vals) >= 2:
        slope, intercept, r, _, _ = linregress(xs, vals)
        log(f"  {label:<22}: extrapolación N→∞ = {intercept:.3f}, R² = {r**2:.3f}, valores = {dict(zip(Ns_arr.tolist(), [round(v, 3) for v in vals]))}")

with open(f'{OUTPUT_DIR}/datos/summary_per_N.json', 'w') as f:
    json.dump({str(k): v for k, v in summary_per_N.items()}, f, indent=2, default=float)

log("\n=== Análisis completado ===")

## 6. Test de estabilidad temporal (idéntico a v1)

In [ ]:
log("\n" + "="*70)
log("TEST 6: ESTABILIDAD TEMPORAL")
log("="*70)

def F_global(points, k=30):
    F_loc = compute_F_local(points, k_neighbors=k)
    return len(points) * F_loc.var()

def mc_step_T0(points, F_func, sigma_pos=0.005, n_steps=500, seed=42):
    rng = np.random.default_rng(seed)
    N = len(points)
    L = 1.0
    F_curr = F_func(points)
    accept = 0
    for step in range(n_steps):
        i = rng.integers(N)
        old_pos = points[i].copy()
        points[i] = (points[i] + sigma_pos * rng.standard_normal(3)) % L
        F_new = F_func(points)
        if F_new < F_curr:
            F_curr = F_new
            accept += 1
        else:
            points[i] = old_pos
    return points, F_curr, accept

stability_results = []
n_replicas_stability = 3
n_mc_steps = 500

for r in range(n_replicas_stability):
    seed = 100 * (r + 1) + 1
    log(f"\n--- Estabilidad réplica {r+1}/{n_replicas_stability}, seed={seed} ---")
    t0 = time.time()
    
    points_init = init_cristal_jiterado(2000, jitter=JITTER, seed=seed)
    F_local_init = compute_F_local(points_init)
    
    Lap_init = build_laplacian_dense(points_init, k_neighbors=30)
    eigvals_init, iprs_init, centers_init, _ = diagonalize_laplacian(Lap_init, 2000, mode='dense')
    
    localized_init = iprs_init > 0.10
    n_loc_init = int(localized_init.sum())
    omegas_init = np.sqrt(np.maximum(eigvals_init, 0))
    omega_loc_init = omegas_init[localized_init]
    centers_loc_init = centers_init[localized_init]
    pos_centers_init = points_init[centers_loc_init]
    
    log(f"  Inicial: {n_loc_init} modos localizados, ⟨ω⟩={omega_loc_init.mean():.3f}")
    
    t_mc = time.time()
    points_evolved, F_final, n_accept = mc_step_T0(points_init.copy(), F_global, n_steps=n_mc_steps, seed=seed+999)
    log(f"  MC: {n_accept}/{n_mc_steps} aceptados, {time.time()-t_mc:.1f}s")
    
    Lap_evolved = build_laplacian_dense(points_evolved, k_neighbors=30)
    eigvals_ev, iprs_ev, centers_ev, _ = diagonalize_laplacian(Lap_evolved, 2000, mode='dense')
    
    localized_ev = iprs_ev > 0.10
    omegas_ev = np.sqrt(np.maximum(eigvals_ev, 0))
    omega_loc_ev = omegas_ev[localized_ev]
    centers_loc_ev = centers_ev[localized_ev]
    pos_centers_ev = points_evolved[centers_loc_ev]
    
    matched_distances = []
    matched_omega_diff = []
    
    for i, (pos_i, omega_i) in enumerate(zip(pos_centers_init, omega_loc_init)):
        diff = pos_centers_ev - pos_i
        diff = diff - np.round(diff)
        dists = np.linalg.norm(diff, axis=-1)
        if len(dists) == 0:
            continue
        nearest = np.argmin(dists)
        matched_distances.append(float(dists[nearest]))
        matched_omega_diff.append(float(omega_loc_ev[nearest] - omega_i))
    
    a = 2000**(-1/3)
    matched_distances_in_a = np.array(matched_distances) / a
    
    stable_frac = float((matched_distances_in_a < 2.0).mean())
    log(f"  Match dist: ⟨d⟩={np.mean(matched_distances_in_a):.2f} a, σ={np.std(matched_distances_in_a):.2f}")
    log(f"  Estable (d<2 spacings): {stable_frac*100:.1f}%")
    
    stability_results.append({
        'replica': r, 'seed': seed,
        'n_loc_init': n_loc_init, 'n_loc_evolved': int(localized_ev.sum()),
        'mean_omega_init': float(omega_loc_init.mean()),
        'mean_omega_evolved': float(omega_loc_ev.mean()),
        'mean_match_dist_in_a': float(np.mean(matched_distances_in_a)),
        'std_match_dist_in_a': float(np.std(matched_distances_in_a)),
        'mean_omega_diff': float(np.mean(matched_omega_diff)),
        'stable_fraction': stable_frac,
        'mc_acceptance': n_accept / n_mc_steps,
    })
    
    del Lap_init, Lap_evolved, eigvals_init, eigvals_ev, iprs_init, iprs_ev
    del centers_init, centers_ev, points_init, points_evolved
    gc.collect()

log("\n=== Resumen Estabilidad ===")
stable_fracs = [s['stable_fraction'] for s in stability_results]
log(f"Fracción estable: {np.mean(stable_fracs)*100:.1f}% ± {np.std(stable_fracs)*100:.1f}%")

with open(f'{OUTPUT_DIR}/datos/test6_stability.json', 'w') as f:
    json.dump(stability_results, f, indent=2)

## 7. Figuras

In [ ]:
log("\nGenerando figuras...")

# Fig 1: Perfil radial + ξ_def vs N
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for N in sorted(all_results.keys()):
    if not all_results[N]:
        continue
    r = all_results[N][0]
    if r['profile_low_median'] and r['profile_high_median']:
        rs = r['shell_centers_in_a']
        axes[0].plot(rs, r['profile_low_median'], 'o-', label=f'N={N} (low)', markersize=8)
        axes[0].plot(rs, r['profile_high_median'], 's-', label=f'N={N} (high)', markersize=8, alpha=0.7)
axes[0].axhline(0, color='black', lw=0.5)
axes[0].set_xlabel('r / a')
axes[0].set_ylabel('mediana(ΔF) en cáscara')
axes[0].set_title('(a) Perfil radial — defectos por percentil')
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

Ns = sorted(summary_per_N.keys())
for label_key, label, marker, color in [
    ('xi_def_low_mean', 'ξ low (blandos)', 'o', 'C0'),
    ('xi_def_high_mean', 'ξ high (duros)', 's', 'C1'),
]:
    vals = [summary_per_N[N].get(label_key) for N in Ns]
    ses = [summary_per_N[N].get(label_key.replace('mean', 'se')) for N in Ns]
    Ns_v = [N for N, v in zip(Ns, vals) if v is not None]
    vals_v = [v for v in vals if v is not None]
    ses_v = [s for v, s in zip(vals, ses) if v is not None]
    if Ns_v:
        axes[1].errorbar(Ns_v, vals_v, yerr=ses_v, fmt=marker+'-', label=label, markersize=10, color=color)
axes[1].set_xscale('log')
axes[1].set_xlabel('N')
axes[1].set_ylabel('ξ_def (en spacings)')
axes[1].set_title('(b) Longitud de localización vs N')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/figuras/fig1_defectos.png', dpi=150, bbox_inches='tight')
plt.close()

# Fig 2: Mobility edge
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for N in sorted(all_results.keys()):
    if not all_results[N]:
        continue
    r = all_results[N][0]
    if r.get('band_centers') and r.get('band_ipr_med'):
        axes[0].semilogy(r['band_centers'], r['band_ipr_med'], 'o-', label=f'N={N}', markersize=8)
axes[0].axhline(1e-3, color='gray', ls=':', label='~1/N')
axes[0].axhline(0.1, color='red', ls='--', label='IPR=0.1')
axes[0].set_xlabel('ω')
axes[0].set_ylabel('IPR mediano')
axes[0].set_title('(a) IPR vs ω')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

mobs = [summary_per_N[N]['mobility_edge_mean'] for N in Ns if summary_per_N[N]['mobility_edge_mean'] is not None]
ses = [summary_per_N[N]['mobility_edge_se'] for N in Ns if summary_per_N[N]['mobility_edge_mean'] is not None]
Ns_v = [N for N in Ns if summary_per_N[N]['mobility_edge_mean'] is not None]
if Ns_v:
    axes[1].errorbar(Ns_v, mobs, yerr=ses, fmt='o-', markersize=10, capsize=5)
axes[1].set_xscale('log')
axes[1].set_xlabel('N')
axes[1].set_ylabel('ω mobility edge')
axes[1].set_title('(b) Mobility edge vs N')
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/figuras/fig2_espectro.png', dpi=150, bbox_inches='tight')
plt.close()

# Fig 3: Enriquecimiento
fig, ax = plt.subplots(1, 1, figsize=(10, 6))
for label_key, label, marker, color in [
    ('enrichment_low_mean', 'Enr low (blandos→CDM)', 'o', 'C0'),
    ('enrichment_high_mean', 'Enr high (duros→bariones)', 's', 'C2'),
]:
    vals = [summary_per_N[N].get(label_key) for N in Ns]
    ses = [summary_per_N[N].get(label_key.replace('mean', 'se')) for N in Ns]
    Ns_v = [N for N, v in zip(Ns, vals) if v is not None]
    vals_v = [v for v in vals if v is not None]
    ses_v = [s for v, s in zip(vals, ses) if v is not None]
    if Ns_v:
        ax.errorbar(Ns_v, vals_v, yerr=ses_v, fmt=marker+'-', label=label, markersize=10, color=color)
ax.axhline(1, color='black', ls=':', label='No enriquecimiento')
ax.set_xscale('log')
ax.set_xlabel('N')
ax.set_ylabel('Enriquecimiento')
ax.set_title('Enriquecimiento de modos localizados en defectos vs N')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/figuras/fig3_enriquecimiento.png', dpi=150, bbox_inches='tight')
plt.close()

# Fig 4: DOS
fig, ax = plt.subplots(1, 1, figsize=(10, 6))
for N in sorted(all_results.keys())[-2:]:
    if not all_results[N]: continue
    r = all_results[N][0]
    if r.get('omega_sample'):
        omegas = np.array(r['omega_sample'])
        iprs = np.array(r['ipr_sample'])
        mask_deloc = iprs < 0.05
        mask_loc = iprs > 0.10
        ax.hist(omegas[mask_deloc], bins=50, alpha=0.4, label=f'Deslocalizados (N={N})', density=True)
        ax.hist(omegas[mask_loc], bins=50, alpha=0.4, label=f'Localizados (N={N})', density=True)
ax.set_xlabel('ω')
ax.set_ylabel('Densidad (normalizada)')
ax.set_title('Densidad de estados por sector')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/figuras/fig4_DOS.png', dpi=150, bbox_inches='tight')
plt.close()

# Fig 5: Estabilidad
if stability_results:
    fig, ax = plt.subplots(1, 1, figsize=(10, 6))
    fracs = [s['stable_fraction'] for s in stability_results]
    dists = [s['mean_match_dist_in_a'] for s in stability_results]
    x = np.arange(len(stability_results))
    width = 0.35
    ax.bar(x - width/2, fracs, width, label='Fracción estable (d<2 spacings)', color='C2')
    ax2 = ax.twinx()
    ax2.bar(x + width/2, dists, width, label='⟨d⟩ centros', color='C1', alpha=0.7)
    ax.set_xticks(x)
    ax.set_xticklabels([f'Réplica {s["replica"]+1}' for s in stability_results])
    ax.set_ylabel('Fracción estable', color='C2')
    ax2.set_ylabel('Distancia (spacings)', color='C1')
    ax.set_title('Estabilidad temporal')
    ax.legend(loc='upper left')
    ax2.legend(loc='upper right')
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/figuras/fig5_estabilidad.png', dpi=150, bbox_inches='tight')
    plt.close()

log("  Figuras guardadas")

## 8. Reporte final

In [ ]:
log("\n" + "="*70)
log("REPORTE FINAL")
log("="*70)

report = {
    'fecha': time.strftime('%Y-%m-%d %H:%M:%S'),
    'parametros': {
        'jitter': JITTER, 'clip_factor': CLIP_FACTOR,
        'defect_percentile': DEFECT_PERCENTILE,
        'sigma_F_target_SIM19': TARGET_SIGMA_F_SIM19,
    },
    'config': config,
    'summary_per_N': {str(k): v for k, v in summary_per_N.items()},
    'stability_results': stability_results,
    'conclusiones': []
}

Ns_done = sorted(summary_per_N.keys())

# Validación inicial
sigma_first = summary_per_N[Ns_done[0]]['F_std_mean'] if Ns_done else None
if sigma_first is not None:
    ratio = sigma_first / TARGET_SIGMA_F_SIM19
    if 0.5 < ratio < 2.0:
        report['conclusiones'].append(
            f"σ(F_local) = {sigma_first:.4f} consistente con SIM 19 ({TARGET_SIGMA_F_SIM19}, ratio={ratio:.2f})")
    else:
        report['conclusiones'].append(
            f"⚠ σ(F_local) = {sigma_first:.4f} no coincide con SIM 19 (ratio={ratio:.2f})")

# Estabilidad de ξ
xis_l = [summary_per_N[N]['xi_def_low_mean'] for N in Ns_done if summary_per_N[N]['xi_def_low_mean'] is not None]
xis_h = [summary_per_N[N]['xi_def_high_mean'] for N in Ns_done if summary_per_N[N]['xi_def_high_mean'] is not None]
if xis_l and xis_h:
    var_l = np.std(xis_l) / np.mean(xis_l) if np.mean(xis_l) > 0 else 0
    var_h = np.std(xis_h) / np.mean(xis_h) if np.mean(xis_h) > 0 else 0
    if var_l < 0.3 and var_h < 0.3:
        report['conclusiones'].append(
            f"ξ_def estable: low={np.mean(xis_l):.2f}±{np.std(xis_l):.2f}, high={np.mean(xis_h):.2f}±{np.std(xis_h):.2f} spacings")
    else:
        report['conclusiones'].append(
            f"ξ_def todavía variable: low varía {var_l*100:.0f}%, high varía {var_h*100:.0f}%")

# Enriquecimiento
enrs_h = [summary_per_N[N]['enrichment_high_mean'] for N in Ns_done 
          if summary_per_N[N]['enrichment_high_mean'] is not None]
enrs_l = [summary_per_N[N]['enrichment_low_mean'] for N in Ns_done 
          if summary_per_N[N]['enrichment_low_mean'] is not None]
if enrs_h:
    report['conclusiones'].append(
        f"Enriquecimiento HIGH: ⟨enr⟩ = {np.mean(enrs_h):.2f} (σ={np.std(enrs_h):.2f}) a través de N")
if enrs_l:
    report['conclusiones'].append(
        f"Enriquecimiento LOW: ⟨enr⟩ = {np.mean(enrs_l):.2f} (σ={np.std(enrs_l):.2f})")

# Estabilidad temporal
if stability_results:
    sf = np.mean([s['stable_fraction'] for s in stability_results])
    report['conclusiones'].append(f"Estabilidad temporal: {sf*100:.0f}% (3 réplicas N=2000, MC=500 pasos)")

log("\n>>> Conclusiones:")
for c in report['conclusiones']:
    log(f"   - {c}")

with open(f'{OUTPUT_DIR}/REPORTE_FINAL.json', 'w') as f:
    json.dump(report, f, indent=2, default=float)

with open(f'{OUTPUT_DIR}/REPORTE_FINAL.txt', 'w') as f:
    f.write("DEE — Test materia v2 (jitter=0.01)\n")
    f.write("="*70 + "\n")
    f.write(f"Fecha: {report['fecha']}\n\n")
    f.write(f"Parámetros: jitter={JITTER}, clip_factor={CLIP_FACTOR}, percentile={DEFECT_PERCENTILE}\n\n")
    f.write("=== Por N ===\n")
    for N in sorted(summary_per_N.keys()):
        s = summary_per_N[N]
        f.write(f"\nN={N} ({s['n_replicas']} réplicas):\n")
        f.write(f"  σ(F_local): {s['F_std_mean']:.4f} ± {s['F_std_se']:.4f}\n")
        f.write(f"  Defectos: low={s['frac_low_mean']*100:.1f}% (P{DEFECT_PERCENTILE}), high={s['frac_high_mean']*100:.1f}%\n")
        f.write(f"  ξ_low: {s['xi_def_low_mean']}, ξ_high: {s['xi_def_high_mean']}\n")
        f.write(f"  Mobility edge ω: {s['mobility_edge_mean']}\n")
        f.write(f"  Enr low: {s['enrichment_low_mean']}, Enr high: {s['enrichment_high_mean']}\n")
    f.write("\n=== Estabilidad ===\n")
    for s in stability_results:
        f.write(f"  Réplica {s['replica']}: estable {s['stable_fraction']*100:.0f}%, d={s['mean_match_dist_in_a']:.2f}\n")
    f.write("\n=== Conclusiones ===\n")
    for c in report['conclusiones']:
        f.write(f"  - {c}\n")

log_handle.close()

import shutil
shutil.make_archive('/content/dee_materia_v2', 'zip', OUTPUT_DIR)
print("\nResultados empaquetados.")

try:
    from google.colab import files
    files.download('/content/dee_materia_v2.zip')
    print("Descarga iniciada.")
except ImportError:
    print("Descarga manual desde /content/dee_materia_v2.zip")

print("\n" + "="*70)
print("COMPLETADO")
print("="*70)